<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Refactoring Spaghetti Code into Clean, Modular Code

## Introduction
When begin coding, it's completely normal to create one long, linear script that “gets the job done.”

But over time, this results in:

- spaghetti code (tangled logic)
- repeated hardcoded values
- difficulty reusing code
- challenging debugging
- no clear structure

Refactoring is the process of restructuring code without changing its behaviour, improving readability, modularity, and maintainability.

## Steps to refactoring: 

### Understand what the current code actually does
Before changing anything:

- Identify the main tasks the script performs.
- Note dependencies, inputs, outputs, and side effects.
- Write down expected behaviour (even informally).
- Add comments to clarify intent if needed.

This forms your map for refactoring.


### Add tests (even basic ones)
You don't need full CI/CD yet — just:

- Smoke tests: “Does the code run end‑to‑end without errors?”
- Golden tests: fixed inputs → fixed expected outputs
- Tests for critical functions or edge cases

Why?

Tests give you confidence to refactor without breaking stuff.

### Improve readability inside the existing script
Without changing the architecture yet:

- Remove dead code
- Improve variable names
- Introduce varaiables instead of hardcoding values
- Format according to PEP8 (Python) or relevant style guide

This makes the next steps a lot easier.

### Extract functions (the key turning point)
Start grouping logic logically:

- One function = one responsibility
- Each function has clear inputs and outputs


Example:
```python
def load_data(path): ...
def clean_data(df): ...
def transform_data(df): ...
def write_output(df, dest): ...
```


***This is the foundation of modular coding.***


### Introduce modular structure
This is where you break the big script into multiple files:

- `utils.py` for shared helpers
- `data_loading.py`
- `data_processing.py`
- `main.py` to orchestrate everything

### Introduce configuration management
Replace hardcoded values with:

- config files (YAML, JSON, TOML)
- environment variables
- parameterised scripts

**This enables scalability and reusability.**

### Introduce logging instead of print()
Logging gives:

- levels (`info`, `debug`, `error`)
- `timestamps`
- ability to save logs to files

**This improves debuggability and production readiness.**

### Introduce error handling
Add:

- `try/except` blocks where appropriate
- custom error messages
- recovery logic if needed

This makes your code resilient.

### Package and structure the project
Depending on the use-case:

- Create a Python package
- Use `__init__.py`
- Add a proper folder layout (src/ structure)

**This is the step before “scripts calling each other” becomes natural and clean.**

### (Optional but recommended) Add automation

- Add a Makefile or task runner
- Add tests + linting into CI/CD pipeline
- Include documentation (README, docstrings, API docs)


**In short, the stages look like:**

- Understand current code
- Add tests
- Clean readability
- Extract functions
- Modularise (yes!)
- Add configuration
- Add logging
- Add error handling
- Package the project
- Add automation & documentation

**Example:**

In [1]:
import findspark
findspark.init()
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("TitanicRefactor").getOrCreate()

pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/train_titanic.csv")
df = spark.createDataFrame(pdf)

avg = df.select(F.mean("Fare")).collect()[0][0]
df = df.withColumn("AboveAverageFare", F.col("Fare") > avg)
df = df.filter(F.col("AboveAverageFare") == True)
df_final = df.toPandas()
df_final.to_csv("titanic_high_fare.csv", index=False)
print("done")
spark.stop()

done


Issues:

- all logic mixed together
- hardcoded file names
- no functions
- no structure for reuse
- not testable

Lets analyse the code: 

- Understand what the current code actually does
- Add tests (even basic ones)
- Improve readability inside the existing script


In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


- Extract functions (the key turning point)


In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd


Done.


## Refactoring

Now lets use the following project structure as a reference: 


```python
titanic_project/
    main.py
    src/
        __init__.py
        config.py
        io_handler.py
        preprocessing.py
        pipeline.py
```


- `config.py`: Central definition of parameters and paths. This script stores configuration variables and constants used across the project. It includes definitions for column names, test sizes, random states, and file names for models and logs, and paths to datasets.

- `io_handler.py`: Reading, writing and basic cleaning of raw data. Initial splitting(e.g., sampling on large datasets). This script is responsible for extracting the raw  dataset and load the processed dataset. It contains functions to read and write the data from/to a specified file path.

- `preprocessing.py`: This script contains functions for data cleaning and preprocessing. It handles tasks such as converting data types, encoding categorical variables, scaling numerical features, and splitting the data into training and testing sets.

- `pipeline.py`: Orchestrating pipeline stages in sequence. This script defines and orchestrates the end to end pipeline for the project. It encapsulates the sequential steps from data loading and preprocessing.

- `main.py`: Entry point for triggering the pipeline.This is the main entry point for the project. It orchestrates the entire process by calling the main pipeline function and handling overall execution flow.


Lets group our code to follow the structure previously mentioned. 

In [ ]:

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
# ==========================================
# SECTION: CONFIG (The source for src/config.py)
# ==========================================


# ==========================================
# SECTION: IO_HANDLER (The source for src/io_handler.py)
# ==========================================
    

# ==========================================
# SECTION: PREPROCESSING (The source for src/preprocessing.py)
# ==========================================


# ==========================================
# SECTION: PIPELINE (The source for src/pipeline.py)
# ==========================================


# ==========================================
# SECTION: MAIN (The source for main.py)
# ==========================================



Done.


## Exercise: From Monolith to Modular Pipeline

### Background

You have been given a "Spaghetti" script designed to process customer churn data. Currently, this script performs several critical Data Engineering tasks in a single, linear file:

* It initializes a distributed computing environment.
* It extracts raw data from a remote cloud URL.
* It cleans the data (type casting and missing value imputation).
* It performs feature engineering and target standardization.
* It exports the final "Gold" dataset to a local directory.

While the script "works," it is difficult to maintain, impossible to unit test, and hardcoded to a specific dataset.

### Your Mission

Your goal is to refactor this monolith into a **Modular Project Structure**. You will evolve the code through a functional cleanup phase and finally "explode" it into a professional directory structure that separates **Configuration**, **Logic**, and **Orchestration**.

---

### Phase 1: The Functional Cleanup

Before splitting the files, you must group the "Spaghetti" logic into named functions. This helps identify the **Single Responsibility** of each block of code.

**Task:** Group your logic into the following sections within a single script:

1. **Configuration Section:** Define all paths and column lists as constants.
2. **IO Handler Section:** Create functions for `extract_from_csv` and `load_to_csv`.
3. **Preprocessing Section:** Create functions for `handle_types`, `impute_values`, and `add_features`.
4. **Pipeline Section:** Create a master function `run_de_pipeline` that calls the others in order.

---

### Phase 2: The Project Refactor

Now that your logic is modular, you will move these sections into a professional folder structure. This decoupling allows different teams to work on different parts of the pipeline simultaneously.

**Task:** Create the following project structure and distribute your code:

* **`src/config.py`**: Move all hardcoded variables here.
* **`src/io_handler.py`**: Move all data reading and writing functions here.
* **`src/preprocessing.py`**: Move all data cleaning and transformation logic here.
* **`src/pipeline.py`**: Move the orchestration function here.
* **`main.py`**: This should be your only entry point. It should import the `SparkSession`, import the `config` and `pipeline` modules, and trigger the execution.

---

### Success Criteria

1. **Zero Hardcoding:** No URLs or Column names should exist in `preprocessing.py` or `main.py` (they must come from `config.py`).
2. **Clean Execution:** Running `python main.py` should trigger the full Spark pipeline and produce a success log.
3. **Traceability (Optional):** Your terminal should show clear `INFO` logs for each stage of the process (Extraction  Preprocessing  Saving).

### Discussion Questions for the End

* If we decided to change our imputation strategy from *Mean* to *Median*, which file would we modify?
* If the S3 URL for the raw data changes, how many files do we need to edit?
* Why do we keep the `SparkSession` initialization in `main.py` instead of inside the `preprocessing` functions?

In [ ]:
##Original code
import findspark
findspark.init()
import pandas as pd
from pyspark.sql import SparkSession, functions as F


# Using .master("local[*]") tells Spark to use all available cores
spark = SparkSession.builder \
        .appName("SpaguettiChurnPipeline") \
        .getOrCreate()

pdf = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/DataSources/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = spark.createDataFrame(pdf)

df = df.withColumn('TotalCharges', F.col('TotalCharges').cast('double'))

for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    mean_val = df.select(F.mean(F.col(col))).collect()[0][0]
    df = df.fillna({col: mean_val})

cat_list = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService']
for col in cat_list:
    mode_val = df.groupby(col).count().orderBy("count", ascending=False).first()[0]
    df = df.fillna({col: mode_val})

df = df.withColumn('MonthlyChargeRatio', F.col('TotalCharges') / (F.col('tenure') + 1))
df = df.withColumn('churn_binary', F.when(F.col('Churn') == 'Yes', 1).when(F.col('Churn') == 'No', 0))

final_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'MonthlyChargeRatio', 'churn_binary']
final_pdf = df.select(final_cols).toPandas()
final_pdf.to_csv("churn_processed.csv", index=False)

spark.stop()